<a href="https://colab.research.google.com/github/the-name-is-madhu/BIS_LAB/blob/main/1BM24CS154__Financial_Portfolio_Optimization_Cuckoo_Search_Algorithm_for_Optimization_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np
import pandas as pd
import math

# ==========================================
# 1. GENERATE HISTORICAL ASSET RETURNS
# ==========================================

np.random.seed(42)

assets = ["Stock_A", "Stock_B", "Stock_C",
          "Stock_D", "Stock_E"]

returns = pd.DataFrame({
    "Stock_A": np.random.normal(0.0008, 0.015, 500),
    "Stock_B": np.random.normal(0.0006, 0.012, 500),
    "Stock_C": np.random.normal(0.0009, 0.020, 500),
    "Stock_D": np.random.normal(0.0005, 0.010, 500),
    "Stock_E": np.random.normal(0.0010, 0.018, 500)
})

# Calculate mean returns and covariance matrix
mean_returns = returns.mean().values
cov_matrix = returns.cov().values

n = len(assets)

# Investment budget
budget = 100000

# Maximum investment weight per asset
max_weight = 0.60

# Risk aversion parameter
risk_lambda = 2.0


# ==========================================
# 2. CUCKOO SEARCH PARAMETERS
# ==========================================

n_nests = 25
max_iterations = 200
Pa = 0.25
alpha = 0.01


# ==========================================
# 3. NORMALIZE PORTFOLIO WEIGHTS
# ==========================================

def normalize_weights(weights):

    weights = np.asarray(weights, dtype=float)

    # Project weights onto constrained simplex
    low = np.min(weights) - max_weight
    high = np.max(weights)

    for _ in range(100):

        mid = (low + high) / 2

        projected = np.clip(
            weights - mid,
            0,
            max_weight
        )

        if projected.sum() > 1:
            low = mid
        else:
            high = mid

    result = np.clip(
        weights - (low + high) / 2,
        0,
        max_weight
    )

    return result / result.sum()


# ==========================================
# 4. FITNESS FUNCTION
# ==========================================

def fitness(weights):

    # Expected portfolio return
    portfolio_return = np.dot(
        weights, mean_returns
    )

    # Portfolio variance
    portfolio_variance = np.dot(
        weights,
        np.dot(cov_matrix, weights)
    )

    # Maximize return and minimize risk
    score = (
        portfolio_return
        - risk_lambda * portfolio_variance
    )

    return score


# ==========================================
# 5. LEVY FLIGHT FUNCTION
# ==========================================

def levy_flight(beta=1.5):

    sigma_u = (
        math.gamma(1 + beta)
        * math.sin(math.pi * beta / 2)
        / (
            math.gamma((1 + beta) / 2)
            * beta
            * 2 ** ((beta - 1) / 2)
        )
    ) ** (1 / beta)

    u = np.random.normal(
        0, sigma_u, n
    )

    v = np.random.normal(
        0, 1, n
    )

    step = u / (
        np.abs(v) ** (1 / beta) + 1e-10
    )

    return step


# ==========================================
# 6. INITIALIZE CUCKOO NESTS
# ==========================================

nests = np.array([
    normalize_weights(np.random.rand(n))
    for _ in range(n_nests)
])

scores = np.array([
    fitness(nest)
    for nest in nests
])

best_index = np.argmax(scores)

best_nest = nests[best_index].copy()
best_score = scores[best_index]


# ==========================================
# 7. MAIN CUCKOO SEARCH ALGORITHM
# ==========================================

for iteration in range(max_iterations):

    for i in range(n_nests):

        # Generate new solution using Levy flight
        step = levy_flight()

        new_solution = (
            nests[i]
            + alpha * step
            * (nests[i] - best_nest)
            + alpha * np.random.randn(n)
        )

        # Apply investment constraints
        new_solution = normalize_weights(
            new_solution
        )

        new_score = fitness(new_solution)

        # Select a random nest
        j = np.random.randint(n_nests)

        # Replace existing nest if better
        if new_score > scores[j]:

            nests[j] = new_solution
            scores[j] = new_score

    # Abandon worst nests
    abandon_count = int(
        Pa * n_nests
    )

    worst_indices = np.argsort(
        scores
    )[:abandon_count]

    for j in worst_indices:

        nests[j] = normalize_weights(
            np.random.rand(n)
        )

        scores[j] = fitness(nests[j])

    # Update best solution
    current_best = np.argmax(scores)

    if scores[current_best] > best_score:

        best_score = scores[current_best]

        best_nest = nests[
            current_best
        ].copy()

    # Display progress
    if (iteration + 1) % 50 == 0:

        print(
            "Iteration:",
            iteration + 1,
            "| Best Fitness:",
            round(best_score, 8)
        )


# ==========================================
# 8. CALCULATE OPTIMAL PORTFOLIO
# ==========================================

optimal_weights = best_nest

portfolio_return = np.dot(
    optimal_weights,
    mean_returns
)

portfolio_variance = np.dot(
    optimal_weights,
    np.dot(cov_matrix, optimal_weights)
)

portfolio_risk = np.sqrt(
    portfolio_variance
)

# Calculate investment allocation
investment = optimal_weights * budget


# ==========================================
# 9. DISPLAY OPTIMAL PORTFOLIO
# ==========================================

print("\n======================================")
print("       OPTIMAL PORTFOLIO RESULTS")
print("======================================")

result = pd.DataFrame({

    "Asset": assets,

    "Optimal Weight (%)":
        optimal_weights * 100,

    "Investment Amount (Rs.)":
        investment

})

print(
    result.to_string(
        index=False,
        formatters={
            "Optimal Weight (%)": "{:.2f}".format,
            "Investment Amount (Rs.)": "{:.2f}".format
        }
    )
)

print("\n======================================")
print("          PERFORMANCE METRICS")
print("======================================")

print(
    "Total Investment: Rs.",
    round(investment.sum(), 2)
)

print(
    "Expected Daily Return:",
    round(portfolio_return, 6)
)

print(
    "Daily Portfolio Variance:",
    round(portfolio_variance, 8)
)

print(
    "Daily Portfolio Risk:",
    round(portfolio_risk, 6)
)

print(
    "Annualized Expected Return (%):",
    round(portfolio_return * 252 * 100, 2)
)

print(
    "Annualized Portfolio Risk (%):",
    round(portfolio_risk * np.sqrt(252) * 100, 2)
)

print(
    "Best Fitness Score:",
    round(best_score, 8)
)

print("\n======================================")
print("       CONSTRAINT VERIFICATION")
print("======================================")

print(
    "Budget Constraint:",
    "Satisfied" if abs(optimal_weights.sum() - 1) < 1e-6
    else "Not Satisfied"
)

print(
    "Maximum Allocation Constraint:",
    "Satisfied" if np.all(optimal_weights <= max_weight + 1e-6)
    else "Not Satisfied"
)

print(
    "Non-negative Weight Constraint:",
    "Satisfied" if np.all(optimal_weights >= 0)
    else "Not Satisfied"
)

print("\nOptimization Completed Successfully!")


Iteration: 50 | Best Fitness: 0.0018996
Iteration: 100 | Best Fitness: 0.0018996
Iteration: 150 | Best Fitness: 0.0018996
Iteration: 200 | Best Fitness: 0.0018996

       OPTIMAL PORTFOLIO RESULTS
  Asset Optimal Weight (%) Investment Amount (Rs.)
Stock_A              14.70                14703.46
Stock_B              21.87                21870.92
Stock_C              60.00                60000.00
Stock_D               3.24                 3236.51
Stock_E               0.19                  189.11

          PERFORMANCE METRICS
Total Investment: Rs. 100000.0
Expected Daily Return: 0.002218
Daily Portfolio Variance: 0.00015905
Daily Portfolio Risk: 0.012611
Annualized Expected Return (%): 55.89
Annualized Portfolio Risk (%): 20.02
Best Fitness Score: 0.0018996

       CONSTRAINT VERIFICATION
Budget Constraint: Satisfied
Maximum Allocation Constraint: Satisfied
Non-negative Weight Constraint: Satisfied

Optimization Completed Successfully!
